# Pan-cancer mutational landscape of 132,181 tumours

This notebook provides two deliberately separate reproducibility routes. The default frozen-output audit reconstructs headline numerical results from prepared analytical tables and displays all nine main and seven supplementary figures without network access or model refitting. The optional full raw-data reconstruction runs the acquisition, curation, statistical-analysis and figure-generation stages only after the documented gated inputs have been supplied. All paths are repository-relative.

## 1. Environment and preflight

Locate the repository, import the packaged result helpers and verify that every required table and figure is present.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'results' / 'tables' / 'cohort_summary.csv').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository')

ROOT = find_repository(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))
from result_summary import (
    MAIN_FIGURES, SUPPLEMENTARY_FIGURES, headline_results, preflight
)

checks = preflight()
assert checks['ready'], checks['missing']
display(pd.Series(checks, name='value').to_frame())
print(f'Repository: {ROOT}')

## 2. Cohort and assay-defined gene coverage

Reconstruct the cohort scale and verify that potential tumour–gene observations partition into assay-covered and unassayed states.

In [ ]:
TABLES = ROOT / 'results' / 'tables'
cohort = pd.read_csv(TABLES / 'cohort_summary.csv').set_index('metric')
value = cohort['value'].astype(int)
assert value['selected_tissue_tumours'] == 132_181
assert value['contributing_studies'] == 367
assert value['reviewed_cancer_families'] == 89
assert value['callable_tumour_gene_observations'] + value['unassayed_tumour_gene_observations'] == value['potential_tumour_gene_observations']
assert value['mutation_positive_tumour_gene_observations'] + value['callable_mutation_negative_observations'] == value['callable_tumour_gene_observations']
unassayed_pct = 100 * value['unassayed_tumour_gene_observations'] / value['potential_tumour_gene_observations']
display(cohort.loc[[
    'input_study_sample_records', 'selected_tissue_tumours', 'contributing_studies',
    'reviewed_cancer_families', 'detailed_oncotree_codes', 'genes_in_compendium',
    'documented_wes_wgs_tumours', 'targeted_panel_tumours',
    'potential_tumour_gene_observations', 'callable_tumour_gene_observations',
    'unassayed_tumour_gene_observations'
]])
print(f'Unassayed tumour–gene observations: {unassayed_pct:.2f}%')

## 3. Recurrence and mutation-level evidence

Summarise assay-aware gene prevalence and its relationship to CMC tier-matched evidence.

In [ ]:
prevalence = pd.read_csv(TABLES / 'gene_prevalence.csv')
evidence = pd.read_csv(TABLES / 'mutation_evidence.csv')
gene_summary = prevalence.merge(evidence, on='gene', how='left')
sentinel_genes = ['TP53', 'KRAS', 'PIK3CA', 'MUC16', 'APC', 'BRAF', 'EGFR', 'IDH1']
display(
    gene_summary.loc[gene_summary.gene.isin(sentinel_genes), [
        'gene', 'nProfiled', 'nMutated', 'freqPct', 'freqCiLowPct',
        'freqCiHighPct', 'pctTierMatched'
    ]].sort_values('freqPct', ascending=False).round(3)
)
top_hotspots = (pd.read_csv(TABLES / 'hotspot_by_cancer.csv')
                .groupby(['gene', 'aa'], as_index=False).nSamples.sum()
                .sort_values('nSamples', ascending=False).head(12))
display(top_hotspots)

## 4. Context-conditioned gene-pair associations

Recover the selected conditioned odds ratios and verify the lung-adenocarcinoma effects highlighted by the analysis.

In [ ]:
pairs = pd.read_csv(TABLES / 'gene_pair_contexts.csv')
luad = pairs.loc[pairs.cancer.eq('LUAD') & pairs.pair.isin(['EGFR-KRAS', 'KEAP1-STK11'])]
assert np.isclose(luad.loc[luad.pair.eq('EGFR-KRAS'), 'noBurden_full_or'].iloc[0], 0.02773275699)
assert np.isclose(luad.loc[luad.pair.eq('KEAP1-STK11'), 'noBurden_full_or'].iloc[0], 10.8291866)
display(luad[[
    'cancer', 'pair', 'noBurden_full_or', 'noBurden_full_ciLow',
    'noBurden_full_ciHigh', 'noBurden_full_fdr', 'leaveTwoOut_full_or',
    'totalBurden_full_or', 'signStableNoBurdenLeaveTwoOut',
    'effectStableNoBurdenLeaveTwoOut'
]].round(5))
display(pairs.sort_values('noBurden_full_fdr').head(14).round(5))

## 5. Pathways, functional phenotypes and network contexts

In [ ]:
pathways = pd.read_csv(TABLES / 'pathway_by_cancer.csv')
functional = pd.read_csv(TABLES / 'functional_crispr.csv')
prism = pd.read_csv(TABLES / 'functional_prism_selected.csv')
network = pd.read_csv(TABLES / 'network_contexts.csv')

print('Pathway combinations:', len(pathways))
display(pathways.sort_values('wesMutationPct', ascending=False).head(12).round(3))
print(f'CRISPR genotypes shown: {len(functional)}; FDR-supported: {functional.adjustedFdr.lt(0.05).sum()}')
display(functional.sort_values('adjustedFdr').head(12).round(4))
print(f'Selected PRISM contexts: {len(prism)}; FDR-supported: {prism.adjustedFdr.lt(0.05).sum()}')
display(prism.sort_values('adjustedFdr').head(12).round(4))
print(f'Network: {len(network)} cancer-specific contexts, {network.pair.nunique()} unique pairs, {network.cancer.nunique()} cancer families')

## 6. Cancer-specific and pan-cancer survival associations

In [ ]:
survival_cancer = pd.read_csv(TABLES / 'survival_cancer_specific.csv')
survival_pan = pd.read_csv(TABLES / 'survival_pan_cancer.csv')
survival_sensitivity = pd.read_csv(TABLES / 'survival_sensitivity.csv')
survival_medians = pd.read_csv(TABLES / 'survival_medians.csv')
survival_piecewise = pd.read_csv(TABLES / 'survival_piecewise_hazard_ratios.csv')
survival_rmst = pd.read_csv(TABLES / 'survival_rmst_differences.csv')
display(survival_cancer.sort_values('jointStateHazardRatio', ascending=False).round(4))
display(survival_pan.sort_values('jointStateHazardRatio', ascending=False).round(4))
display(survival_sensitivity.sort_values(['context', 'model']).round(4))
display(survival_piecewise.loc[survival_piecewise.context.isin(['KEAP1–STK11 (LUAD)', 'KRAS–TP53 (PAAD)'])].round(4))
display(survival_rmst.loc[survival_rmst.context.isin(['KEAP1–STK11 (LUAD)', 'KRAS–TP53 (PAAD)'])].round(4))
display(survival_medians.loc[survival_medians.cancer.isin(['LUAD', 'PAAD'])].round(2))

## 7. Unified numerical summary

In [ ]:
summary = headline_results()
print(json.dumps(summary, indent=2))

## 8. Main figures

In [ ]:
for number, filename in enumerate(MAIN_FIGURES, start=1):
    display(Markdown(f'### Figure {number}'))
    display(Image(filename=str(ROOT / 'results' / 'figures' / filename), width=1100))

## 9. Supplementary figures

In [ ]:
for filename in SUPPLEMENTARY_FIGURES:
    number = filename.split('_', 1)[0].removeprefix('figureS')
    display(Markdown(f'### Supplementary Figure {number}'))
    display(Image(filename=str(ROOT / 'results' / 'figures' / 'supplementary' / filename), width=1100))

## 10. Regenerate the complete analysis and figures

The cell below exposes the gated end-to-end route from within this notebook. Supply the resources listed in `data/external/README.md`, confirm the manifest gate with `python src/run_all.py --dry-run`, set `RUN_FULL_ANALYSIS` to `True`, and run the cell. The reconstruction retrieves public cBioPortal records, rebuilds the curated cohort and analytical tables, refits the statistical models, and regenerates the nine main and seven supplementary figures.

The default is `False` so that opening or testing the notebook does not initiate a large data retrieval. The same route can be launched from a terminal with `python src/run_all.py`; `python src/run_pipeline.py` is reserved for the fast frozen-output audit.

In [ ]:
RUN_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    command = [sys.executable, str(ROOT / 'src' / 'run_all.py')]
    subprocess.run(command, cwd=ROOT, check=True)
    refreshed = preflight()
    assert refreshed['ready'], refreshed['missing']
    display(pd.Series(refreshed, name='value').to_frame())
    print('Complete analysis and all figures regenerated successfully.')
else:
    print('Prepared-output route complete. Set RUN_FULL_ANALYSIS = True to rebuild the complete analysis and figures.')